# 13 — CNN profunda con remuestreo físico + pos_weight/focal loss (ambas estrategias combinadas)

Combina las dos estrategias de balanceo probadas por separado en 04 (pos_weight + focal loss sobre los datos originales) y 04b (remuestreo físico con pérdida estándar): aquí se entrena sobre el conjunto de entrenamiento ya remuestreado (oversampling HYP / undersampling NORM, idéntico a 04b) usando además la pérdida ponderada con pos_weight y modulación focal.

## 1. Entorno, semillas y librerías

In [1]:
import sys, os, random, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import ast

import wfdb
from scipy import signal as sp_signal
from scipy.signal import sosfiltfilt, iirnotch, tf2sos
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.regularizers import l2

print(sys.executable)
print("TensorFlow:", tf.__version__)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

C:\Users\Usuario\anaconda3\envs\tfg\python.exe
TensorFlow: 2.16.1


## 2. Configuración general

In [2]:
DATA_DIR  = Path(r"C:\Users\Usuario\Desktop\TFG\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
CACHE_DIR = DATA_DIR / "cache_v2" / "con_filtrado"

MODEL_DIR = Path("./outputs/models"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR   = Path("./outputs/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_CSV = MODEL_DIR / "model_comparison.csv"

FS = 100
SUPER_CLASSES = ["NORM", "MI", "STTC", "HYP", "CD"]
CLASS_MAP_HUMAN = {
    "NORM": "Normal", "MI": "Infarto", "STTC": "Isquemia/ST-T",
    "HYP": "Hipertrofia", "CD": "Conducción (bloqueos)",
}

BANDPASS_LOW  = 0.5
BANDPASS_HIGH = 40.0
NOTCH_FREQ    = 50.0
NOTCH_Q       = 30.0

STFT_NPERSEG  = 64
STFT_NOVERLAP = 48
SPEC_SHAPE    = (33, 64)
L2_REG        = 1e-4
GAMMA         = 1.0

TARGET_HYP  = 4000
TARGET_NORM = 4000

## 3. Carga de metadatos y etiquetas multi-etiqueta


In [3]:
df  = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
df.columns  = df.columns.str.strip()
scp.columns = scp.columns.str.strip()

scp_super = (scp[scp["diagnostic_class"].notna()][["diagnostic_class"]]
             .rename(columns={"diagnostic_class": "superclass"}))

def extract_superclasses(scp_codes_str):
    if pd.isna(scp_codes_str):
        return []
    try:
        d = ast.literal_eval(scp_codes_str)
        if not isinstance(d, dict):
            return []
    except Exception:
        return []
    return list({scp_super.loc[k, "superclass"] for k in d.keys() if k in scp_super.index})

def multilabel_vector(classes):
    s = set(classes)
    return np.array([1 if c in s else 0 for c in SUPER_CLASSES], dtype=np.int32)

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)
df = df[df["superclasses"].apply(len) > 0].reset_index()
df["target_vec"] = df["superclasses"].apply(multilabel_vector)

## 4. Partición train/validación/test

Idéntico al resto de la serie.

In [4]:
train_df = df[df["strat_fold"].between(1, 8)].reset_index(drop=True)
val_df   = df[df["strat_fold"] == 9].reset_index(drop=True)
test_df  = df[df["strat_fold"] == 10].reset_index(drop=True)

assert set(train_df["patient_id"]) & set(val_df["patient_id"]) == set()
assert set(train_df["patient_id"]) & set(test_df["patient_id"]) == set()
assert set(val_df["patient_id"])   & set(test_df["patient_id"]) == set()

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))

Train: 17084  Val: 2146  Test: 2158


## 5. Preprocesado, STFT y caché en disco


In [5]:
def is_valid_signal(x, flat_std_thresh=1e-6):
    if np.any(np.isnan(x)) or np.any(np.isinf(x)):
        return False
    return np.std(x) >= flat_std_thresh

_BP_SOS    = sp_signal.butter(4, [BANDPASS_LOW / (FS/2), BANDPASS_HIGH / (FS/2)], btype="band", output="sos")
_NOTCH_SOS = tf2sos(*iirnotch(NOTCH_FREQ / (FS/2), NOTCH_Q))

def clean_lead(x):
    x = np.asarray(x, dtype=np.float64)
    if not is_valid_signal(x):
        return None
    x = sosfiltfilt(_BP_SOS, x)
    x = sosfiltfilt(_NOTCH_SOS, x)
    mu, sigma = x.mean(), x.std()
    if sigma < 1e-8:
        return None
    return ((x - mu) / sigma).astype(np.float32)

def preprocess_ecg(raw_signal):
    cleaned = np.zeros_like(raw_signal, dtype=np.float32)
    n_bad = 0
    for i in range(raw_signal.shape[1]):
        c = clean_lead(raw_signal[:, i])
        cleaned[:, i] = c if c is not None else 0.0
        n_bad += (c is None)
    return None if n_bad > raw_signal.shape[1] // 2 else cleaned

def lead_to_spectrogram(lead_signal, fs=FS, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP, out_shape=SPEC_SHAPE):
    _, _, Zxx = sp_signal.stft(lead_signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
    mag = np.log1p(np.abs(Zxx))
    F, T = out_shape
    mag = mag[:F, :T]
    if mag.shape != (F, T):
        pad = np.zeros((F, T), dtype=np.float32)
        pad[:mag.shape[0], :mag.shape[1]] = mag
        mag = pad
    mn, mx = mag.min(), mag.max()
    if mx - mn > 1e-8:
        mag = (mag - mn) / (mx - mn)
    return mag.astype(np.float32)

def ecg_to_tensor(signal_2d):
    specs = [lead_to_spectrogram(signal_2d[:, i]) for i in range(signal_2d.shape[1])]
    return np.stack(specs, axis=-1)

def _cache_path(rel_path):
    h = hashlib.md5(rel_path.encode()).hexdigest()
    return CACHE_DIR / f"{h}.npy"

def get_tensor_cached(rel_path):
    cp = _cache_path(rel_path)
    if cp.exists():
        return np.load(cp)
    raw, _ = wfdb.rdsamp(str(DATA_DIR / rel_path))
    clean = preprocess_ecg(raw)
    if clean is None:
        return None
    tensor = ecg_to_tensor(clean)
    np.save(cp, tensor)
    return tensor

def build_split(df_split):
    Xs, Ys = [], []
    for _, row in df_split.iterrows():
        t = get_tensor_cached(row["filename_lr"])
        if t is None:
            continue
        Xs.append(t)
        Ys.append(row["target_vec"])
    return np.stack(Xs).astype(np.float32), np.stack(Ys).astype(np.float32)

## 6. Arquitectura

In [6]:
def build_model_deep(input_shape, n_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(64, activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation="sigmoid")(x)
    return models.Model(inputs, outputs)

## 7. Remuestreo del conjunto de entrenamiento


In [7]:
def has_class(df_, cls):
    idx = SUPER_CLASSES.index(cls)
    return df_["target_vec"].apply(lambda v: v[idx] == 1)

def is_norm_only(row):
    v = row["target_vec"]
    return v[SUPER_CLASSES.index("NORM")] == 1 and v.sum() == 1

train_df_rs = train_df.copy()

hyp_mask = has_class(train_df_rs, "HYP")
n_hyp = hyp_mask.sum()
print(f"HYP antes: {n_hyp}")
if n_hyp < TARGET_HYP:
    extra = train_df_rs[hyp_mask].sample(n=TARGET_HYP - n_hyp, replace=True, random_state=SEED)
    train_df_rs = pd.concat([train_df_rs, extra], ignore_index=True)

norm_only_mask = train_df_rs.apply(is_norm_only, axis=1)
n_norm_only = norm_only_mask.sum()
print(f"Normal-solo antes: {n_norm_only}")
if n_norm_only > TARGET_NORM:
    keep_norm = train_df_rs[norm_only_mask].sample(n=TARGET_NORM, random_state=SEED)
    train_df_rs = pd.concat([train_df_rs[~norm_only_mask], keep_norm], ignore_index=True)

train_df_rs = train_df_rs.sample(frac=1, random_state=SEED).reset_index(drop=True)
print("Tamaño final de entrenamiento remuestreado:", len(train_df_rs))
print("HYP después:", has_class(train_df_rs, "HYP").sum())

HYP antes: 2119
Normal-solo antes: 7243
Tamaño final de entrenamiento remuestreado: 15722
HYP después: 4000


## 8. Construcción de los tensores (train remuestreado, val y test sin tocar)

In [8]:
X_train, Y_train = build_split(train_df_rs)
X_val,   Y_val   = build_split(val_df)
X_test,  Y_test  = build_split(test_df)
print(X_train.shape, X_val.shape, X_test.shape)

(15722, 33, 64, 12) (2146, 33, 64, 12) (2158, 33, 64, 12)


## 9. pos_weight (sobre el train YA remuestreado) y pérdida focal ponderada

Se calcula `pos_weight` sobre `Y_train` remuestreado, no sobre el original: tras el remuestreo, el desbalanceo ya es menor, así que los pesos resultantes serán más moderados.

In [9]:
pos_counts = Y_train.sum(axis=0)
neg_counts = len(Y_train) - pos_counts
pos_weight = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 10.0)
print("pos_weight por clase (sobre train remuestreado):", dict(zip(SUPER_CLASSES, pos_weight.round(2))))

pos_weight_tf = tf.constant(pos_weight, dtype=tf.float32)

def focal_weighted_bce(y_true, y_pred, gamma=GAMMA):
    """Entropía cruzada binaria ponderada por clase (pos_weight) y modulada por dificultad (focal, gamma)."""
    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred, eps, 1 - eps)
    pos_term = pos_weight_tf * y_true * tf.pow(1 - y_pred, gamma) * tf.math.log(y_pred)
    neg_term = (1 - y_true) * tf.pow(y_pred, gamma) * tf.math.log(1 - y_pred)
    return tf.reduce_mean(-(pos_term + neg_term))

pos_weight por clase (sobre train remuestreado): {'NORM': 2.61, 'MI': 2.17, 'STTC': 1.98, 'HYP': 2.93, 'CD': 2.54}


## 10. Entrenamiento

In [10]:
model = build_model_deep(X_train.shape[1:], len(SUPER_CLASSES))
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=focal_weighted_bce,
              metrics=[tf.keras.metrics.AUC(name="auc", multi_label=True)])

cbs = [
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", patience=4),
]

history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val),
                     epochs=60, batch_size=32, callbacks=cbs, verbose=1)

Epoch 1/60
492/492 ━━━━━━━━━━━━━━━━━━━━ 42s 75ms/step - auc: 0.7666 - loss: 0.4270 - val_auc: 0.8339 - val_loss: 0.3758 - learning_rate: 0.0010
Epoch 2/60
492/492 ━━━━━━━━━━━━━━━━━━━━ 37s 74ms/step - auc: 0.8260 - loss: 0.3803 - val_auc: 0.8427 - val_loss: 0.3472 - learning_rate: 0.0010
Epoch 3/60
492/492 ━━━━━━━━━━━━━━━━━━━━ 37s 74ms/step - auc: 0.8420 - loss: 0.3651 - val_auc: 0.8582 - val_loss: 0.3403 - learning_rate: 0.0010
Epoch 4/60
492/492 ━━━━━━━━━━━━━━━━━━━━ 39s 79ms/step - auc: 0.8509 - loss: 0.3555 - val_auc: 0.8634 - val_loss: 0.3299 - learning_rate: 0.0010
Epoch 5/60
492/492 ━━━━━━━━━━━━━━━━━━━━ 37s 76ms/step - auc: 0.8570 - loss: 0.3494 - val_auc: 0.8659 - val_loss: 0.3241 - learning_rate: 0.0010
Epoch 6/60
492/492 ━━━━━━━━━━━━━━━━━━━━ 38s 77ms/step - auc: 0.8618 - loss: 0.3436 - val_auc: 0.8651 - val_loss: 0.3209 - learning_rate: 0.0010
Epoch 7/60
492/492 ━━━━━━━━━━━━━━━━━━━━ 38s 77ms/step - auc: 0.8664 - loss: 0.3394 - val_auc: 0.8704 - val_loss: 0.3473 - learning_rate:

## 11. Evaluación sobre test (umbral fijo 0.5)


In [11]:
test_probs = model.predict(X_test, verbose=0)
test_pred = (test_probs >= 0.5).astype(int)

metrics_rows = []
for i, c in enumerate(SUPER_CLASSES):
    y_t, y_p, y_prob = Y_test[:, i], test_pred[:, i], test_probs[:, i]
    metrics_rows.append({
        "Clase": CLASS_MAP_HUMAN[c],
        "Accuracy": accuracy_score(y_t, y_p),
        "Precision": precision_score(y_t, y_p, zero_division=0),
        "Recall": recall_score(y_t, y_p, zero_division=0),
        "F1": f1_score(y_t, y_p, zero_division=0),
        "MAE": np.mean(np.abs(y_prob - y_t)),
    })
metrics_df = pd.DataFrame(metrics_rows)
macro_row = metrics_df.drop(columns="Clase").mean()
metrics_df = pd.concat([metrics_df, pd.DataFrame([{"Clase": "Macro", **macro_row}])], ignore_index=True)
metrics_df

,Clase,Accuracy,Precision,Recall,F1,MAE
0,Normal,0.839203,0.764151,0.925234,0.837013,0.258571
1,Infarto,0.827618,0.677291,0.618182,0.646388,0.318824
2,Isquemia/ST-T,0.839203,0.624286,0.838772,0.715807,0.269277
3,Hipertrofia,0.753475,0.301471,0.782443,0.435244,0.360605
4,Conducción (bloqueos),0.849861,0.667969,0.689516,0.678571,0.292185
5,Macro,0.821872,0.607033,0.770829,0.662605,0.299892


## 12. Registro en la tabla comparativa común

In [12]:
NOTEBOOK_NAME = "04c_cnn_profunda_remuestreo_balanceo"
PREP_VERSION_LABEL = "con_filtrado"

model.save(MODEL_DIR / f"{NOTEBOOK_NAME}.keras")

macro_row_dict = metrics_df[metrics_df["Clase"] == "Macro"].drop(columns="Clase").iloc[0].to_dict()
result_row = pd.DataFrame([{
    "Notebook": NOTEBOOK_NAME,
    "Preprocesado": PREP_VERSION_LABEL,
    "Parametros_entrenables": model.count_params(),
    **macro_row_dict,
}])

if COMPARISON_CSV.exists():
    comparison = pd.read_csv(COMPARISON_CSV, sep=None, engine="python")
    comparison = comparison[comparison["Notebook"] != NOTEBOOK_NAME]
    comparison = pd.concat([comparison, result_row], ignore_index=True)
else:
    comparison = result_row
comparison.to_csv(COMPARISON_CSV, index=False)
comparison

,Notebook,Preprocesado,Parametros_entrenables,Accuracy,Precision,Recall,F1,MAE
0,01_cnn_simple_sin_filtrado,sin_filtrado,3653,0.823077,0.760360,0.346851,0.413249,0.254433
1,02_cnn_simple_con_filtrado,con_filtrado,3653,0.829472,0.733107,0.387682,0.457371,0.241108
2,03_cnn_profunda_con_filtrado,con_filtrado,105317,0.854310,0.733758,0.591347,0.630620,0.178831
3,04_cnn_profunda_balanceo,con_filtrado,105317,0.810936,0.577661,0.792424,0.656731,0.286431
4,05_cnn_profunda_balanceo_umbral,con_filtrado,105317,0.840871,0.637769,0.722755,0.672034,0.322418
5,06_random_forest,features_clasicas,1323262,0.833920,0.696762,0.387704,0.467948,0.258019
6,07_xgboost,features_clasicas,125190,0.837442,0.708547,0.499513,0.567501,0.200939
7,08_regresion_logistica,features_clasicas,665,0.825301,0.674439,0.443341,0.504480,0.239834
8,04b_cnn_profunda_remuestreo,con_filtrado,105317,0.852734,0.708761,0.597138,0.637073,0.191559
9,04c_cnn_profunda_remuestreo_balanceo,con_filtrado,105317,0.821872,0.607033,0.770829,0.662605,0.299892


## 13. Comparación directa: 04 (pérdida) vs. 04b (remuestreo) vs. 04c (ambas)

In [13]:
comparison_all = pd.read_csv(COMPARISON_CSV, sep=None, engine="python")
cols = ["Notebook", "Accuracy", "Precision", "Recall", "F1", "MAE"]
nombres = ["04_cnn_profunda_balanceo", "04b_cnn_profunda_remuestreo", "04c_cnn_profunda_remuestreo_balanceo"]
comparison_all[comparison_all["Notebook"].isin(nombres)][cols]

,Notebook,Accuracy,Precision,Recall,F1,MAE
3,04_cnn_profunda_balanceo,0.810936,0.577661,0.792424,0.656731,0.286431
8,04b_cnn_profunda_remuestreo,0.852734,0.708761,0.597138,0.637073,0.191559
9,04c_cnn_profunda_remuestreo_balanceo,0.821872,0.607033,0.770829,0.662605,0.299892
